# 🎓 SmartEvent Strathmore University — Machine Learning Pipeline (Google Colab)
**Author:** Elvis Wanyama Wafuke (154210)  
**Supervisor:** Mr. Allan Vikiru  
**Institution:** Strathmore University — School of Computing and Engineering Sciences (SCES)  

---
### Pipeline Stages Included in This Notebook:
1. **Environment Setup & Repository Sync** (Pulls the `smartevent` ML package or runs standalone)
2. **Stage 1 — Kaggle Dataset Audit & Leakage Removal** (`kaggle_prep.py`)
3. **Stage 2 — Synthetic Strathmore University Event Registration Generation** (`synthetic.py`)
4. **Stage 3 — Deterministic Data Cleaning & Imputation Prep** (`clean.py`)
5. **Stage 4 & 5 — Chronological Event Split (560 Train / 140 Future Test Events) & Feature Engineering**
6. **Stage 6 & 7 — 5-Fold GroupKFold Model Comparison & 5-Stage Logistic Regression Refinement**
7. **Stage 8 — Inline Evaluation Visualizations (ROC Curves, Calibration, Feature Coefficients, Headcount Scatter)**
8. **Live Interactive Attendance & Africa's Talking Priority SMS Boost Prediction**

In [ ]:
# =============================================================================
# CELL 1: Clone SmartEvent Repository & Install Dependencies in Google Colab
# =============================================================================
import os
import sys

if not os.path.exists('/content/smartevent') and 'google.colab' in sys.modules:
    !git clone https://github.com/wafuke/smartevent.git /content/smartevent

if os.path.exists('/content/smartevent/ml-service'):
    os.chdir('/content/smartevent/ml-service')
    if '/content/smartevent/ml-service' not in sys.path:
        sys.path.insert(0, '/content/smartevent/ml-service')

!pip install -q scikit-learn pandas numpy matplotlib joblib
print('Working directory:', os.getcwd())

In [ ]:
# =============================================================================
# CELL 2: Stage 1, 2 & 3 — Data Audit, Synthetic Generation & Data Cleaning
# =============================================================================
import pandas as pd
from smartevent_ml import kaggle_prep, synthetic, clean

print('--- Running Stage 1: Kaggle Benchmark Audit & Leakage Removal ---')
kaggle_prep.run()

print('\n--- Running Stage 2: Synthetic Strathmore Campus Dataset Generation ---')
synthetic.run()

print('\n--- Running Stage 3: Deterministic Data Cleaning ---')
clean.run()

df_clean = pd.read_csv('data/processed/clean_registrations.csv')
print(f'\nCleaned Strathmore Registrations Dataset Shape: {df_clean.shape}')
df_clean.head(10)

In [ ]:
# =============================================================================
# CELL 3: Inspect Cleaned Feature Distributions & Missingness Audit
# =============================================================================
summary_df = pd.DataFrame({
    'Data Type': df_clean.dtypes.astype(str),
    'Missing Values': df_clean.isna().sum(),
    'Missing (%)': (df_clean.isna().mean() * 100).round(2),
    'Unique Values': df_clean.nunique()
})
display(summary_df)

print('\nShow-Up Rate by SMS Priority Tier (1=Low, 2=Medium, 3=High):')
display(df_clean.groupby('sms_priority')['attended'].agg(['count', 'mean']).rename(columns={'mean': 'turnout_rate'}))

In [ ]:
# =============================================================================
# CELL 4: Stage 4 to 8 — Train & Compare Models + Refine Logistic Regression
# =============================================================================
from smartevent_ml import train

results = train.run()

print('\n=== 5-Fold GroupKFold Cross-Validation Model Comparison ===')
cv_df = pd.DataFrame(results['cv_comparison']).T
display(cv_df[['roc_auc_mean', 'roc_auc_std', 'f1_mean', 'brier_mean']])

print('\n=== 5-Stage Iterative Refinement of Logistic Regression ===')
ref_df = pd.DataFrame(results['refinement_stages']).T
display(ref_df)

In [ ]:
# =============================================================================
# CELL 5: Display Generated Evaluation Charts Inline in Google Colab
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_files = [
    ('ROC Curves Comparison', 'reports/figures/roc_curves.png'),
    ('Probability Calibration Curve', 'reports/figures/calibration.png'),
    ('Logistic Regression Feature Coefficients', 'reports/figures/coefficients.png'),
    ('Event-Level Predicted vs Actual Headcount (140 Future Events)', 'reports/figures/event_headcount.png')
]

for title, path in fig_files:
    if os.path.exists(path):
        plt.figure(figsize=(10, 6))
        img = mpimg.imread(path)
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=13, fontweight='bold')
        plt.show()

In [ ]:
# =============================================================================
# CELL 6: Interactive Inference — Predict Turnout for a Strathmore Event
# =============================================================================
import joblib

artifact = joblib.load('models/attendance_model.joblib')
pipeline = artifact['pipeline']
print('Loaded Trained Model:', type(pipeline.named_steps['clf']).__name__)
print('Optimal Probability Threshold:', artifact['optimal_threshold'])
print('Event-Level Test MAE:', round(artifact['metrics']['event_metrics']['mae_attendees'], 2), 'attendees')